# Deep Q-Learning for Atari

In [1]:
%matplotlib inline

import itertools
import os
import random
import sys
from collections import deque, namedtuple

import numpy as np
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
try:
    from torch.utils.tensorboard import SummaryWriter
except ImportError:  # tensorboard package not installed; logging is optional
    SummaryWriter = None
    print("tensorboard not found - summaries disabled. `pip install tensorboard` to enable.")

if "../" not in sys.path:
    sys.path.append("../")

try:
    from lib import plotting
except ImportError:
    plotting = None

# EpisodeStats is the only thing we need from lib.plotting, so define it locally
# as a fallback to keep this notebook self-contained.
EpisodeStats = namedtuple("EpisodeStats", ["episode_lengths", "episode_rewards"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


## Environment

The original used `gym.envs.make("Breakout-v0")` and the old 4-tuple `step` API. Modern
Gymnasium returns a 5-tuple from `step` and an `(obs, info)` pair from `reset`, so the two
helpers below normalise whichever version you have installed. If you're on Gymnasium you'll
also need the ALE ROMs: `pip install "gymnasium[atari,accept-rom-license]"`.

In [2]:
import gymnasium as gym
import ale_py
gym.register_envs(ale_py)

ENV_ID = "ALE/Pong-v5"

In [3]:
class NoopResetEnv(gym.Wrapper):
    """Sample initial states by taking a random number of NOOPs on reset."""

    def __init__(self, env, noop_max=30):
        super().__init__(env)
        self.noop_max = noop_max
        assert env.unwrapped.get_action_meanings()[0] == "NOOP"

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        for _ in range(np.random.randint(1, self.noop_max + 1)):
            obs, _, terminated, truncated, info = self.env.step(0)
            if terminated or truncated:
                obs, info = self.env.reset(**kwargs)
        return obs, info

In [4]:
# render_mode="rgb_array" is required by the video wrapper further down. It does not
# change the observations we train on, only what env.render() returns.
try:
    env = gym.make(ENV_ID, render_mode="rgb_array")
except TypeError:  # gym < 0.26 has no render_mode argument
    env = gym.make(ENV_ID)

env = NoopResetEnv(env, noop_max=8)
env.metadata["render_fps"] = 30

def env_reset(env):
    """Returns just the observation, on either the old or new Gym API."""
    out = env.reset()
    return out[0] if isinstance(out, tuple) else out


def env_step(env, action):
    """Returns (obs, reward, done, info) on either the old or new Gym API."""
    out = env.step(action)
    if len(out) == 5:
        obs, reward, terminated, truncated, info = out
        return obs, reward, bool(terminated or truncated), info
    return out

In [5]:
# Atari Actions: 0 (noop), 1 (fire), 2 (left) and 3 (right) are valid actions
VALID_ACTIONS = [0, 1, 2, 3]

## State processing

No graph, no session — just NumPy. The index arithmetic reproduces
`tf.image.ResizeMethod.NEAREST_NEIGHBOR` with `align_corners=False`, and the grayscale
weights are the ones `tf.image.rgb_to_grayscale` uses.

In [6]:
class StateProcessor:
    """
    Processes a raw Atari image: crops it, converts to grayscale and resizes to 84x84.
    """

    # Same luma coefficients tf.image.rgb_to_grayscale uses.
    _RGB_WEIGHTS = np.array([0.2989, 0.5870, 0.1140], dtype=np.float32)

    def __init__(self, crop=(34, 0, 160, 160), out_size=(84, 84)):
        offset_h, offset_w, target_h, target_w = crop
        self._rows = slice(offset_h, offset_h + target_h)
        self._cols = slice(offset_w, offset_w + target_w)
        # Nearest-neighbour sampling indices, precomputed once.
        self._row_idx = (np.arange(out_size[0]) * target_h // out_size[0]).astype(np.intp)
        self._col_idx = (np.arange(out_size[1]) * target_w // out_size[1]).astype(np.intp)

    def process(self, state):
        """
        Args:
            state: A [210, 160, 3] Atari RGB State

        Returns:
            A processed [84, 84] uint8 array of grayscale values.
        """
        gray = state[self._rows, self._cols].astype(np.float32) @ self._RGB_WEIGHTS
        resized = gray[self._row_idx[:, None], self._col_idx]
        return np.clip(np.round(resized), 0, 255).astype(np.uint8)

## The Q-value estimator

`QNetwork` is the module; `Estimator` wraps it with the same `predict` / `update` interface
the original had, minus the `sess` argument. It owns its optimizer, its global step counter
and its TensorBoard writer, so the two networks stay as independent as the two variable
scopes were.

Normalisation by 255 lives inside `forward`, which lets the replay buffer hold `uint8`
frames — a 500k-transition buffer is ~28 GB as `uint8` and four times that as `float32`.

In [7]:
class QNetwork(nn.Module):
    """Q-Value estimator network (Nature DQN architecture).

    Used for both the Q-Network and the Target Network.
    Input is uint8 of shape [N, 4, 84, 84]; output is [N, num_actions].
    """

    def __init__(self, num_actions=len(VALID_ACTIONS), in_channels=4):
        super().__init__()
        # To match the TF original's SAME padding instead, use padding=2 / 1 / 1 here
        # (with an nn.ZeroPad2d((1, 2, 1, 2)) before conv2) and 64 * 11 * 11 below.
        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=8, stride=4)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, stride=1)
        self.fc1 = nn.Linear(64 * 7 * 7, 512)
        self.fc2 = nn.Linear(512, num_actions)

    def forward(self, x):
        x = x.float() / 255.0
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = F.relu(self.fc1(torch.flatten(x, start_dim=1)))
        return self.fc2(x)


class Estimator:
    """Wraps a QNetwork with its optimizer, step counter and summary writer."""

    def __init__(self, scope="estimator", summaries_dir=None,
                 num_actions=len(VALID_ACTIONS), device=device):
        self.scope = scope
        self.device = device
        self.model = QNetwork(num_actions).to(device)
        # Optimizer parameters from the original paper.
        self.optimizer = optim.RMSprop(
            self.model.parameters(), lr=0.00025, alpha=0.99, eps=1e-6, momentum=0.95)
        self.global_step = 0

        self.summary_writer = None
        if summaries_dir and SummaryWriter is not None:
            summary_dir = os.path.join(summaries_dir, "summaries_{}".format(scope))
            os.makedirs(summary_dir, exist_ok=True)
            self.summary_writer = SummaryWriter(summary_dir)

    def _to_tensor(self, s):
        return torch.as_tensor(np.asarray(s), dtype=torch.uint8, device=self.device)

    @torch.no_grad()
    def predict(self, s):
        """
        Predicts action values.

        Args:
          s: State input of shape [batch_size, 4, 84, 84]

        Returns:
          Array of shape [batch_size, NUM_VALID_ACTIONS] containing the estimated
          action values.
        """
        self.model.eval()
        return self.model(self._to_tensor(s)).cpu().numpy()

    def update(self, s, a, y):
        """
        Updates the estimator towards the given targets.

        Args:
          s: State input of shape [batch_size, 4, 84, 84]
          a: Chosen actions of shape [batch_size]
          y: Targets of shape [batch_size]

        Returns:
          The calculated loss on the batch.
        """
        self.model.train()
        states = self._to_tensor(s)
        actions = torch.as_tensor(np.asarray(a), dtype=torch.long, device=self.device)
        targets = torch.as_tensor(np.asarray(y), dtype=torch.float32, device=self.device)

        predictions = self.model(states)
        # Get the predictions for the chosen actions only
        action_predictions = predictions.gather(1, actions.unsqueeze(1)).squeeze(1)

        # Calculate the loss. Swap in F.smooth_l1_loss for Huber / gradient clipping.
        losses = (targets - action_predictions).pow(2)
        loss = losses.mean()

        self.optimizer.zero_grad(set_to_none=True)
        loss.backward()
        self.optimizer.step()
        self.global_step += 1

        if self.summary_writer:
            step = self.global_step
            self.summary_writer.add_scalar("loss", loss.item(), step)
            self.summary_writer.add_histogram("loss_hist", losses.detach(), step)
            self.summary_writer.add_histogram("q_values_hist", predictions.detach(), step)
            self.summary_writer.add_scalar("max_q_value", predictions.max().item(), step)

        return loss.item()

In [8]:
# For Testing....

torch.manual_seed(0)

e = Estimator(scope="test")
sp = StateProcessor()

# Example observation batch
observation = env_reset(env)

observation_p = sp.process(observation)
observation = np.stack([observation_p] * 4, axis=0)   # [4, 84, 84], channels first
observations = np.array([observation] * 2)            # [2, 4, 84, 84]

# Test Prediction
print(e.predict(observations))

# Test training step
y = np.array([10.0, 10.0])
a = np.array([1, 3])
print(e.update(observations, a, y))

[[ 0.0474164   0.01683912 -0.03499081  0.00905719]
 [ 0.0474164   0.01683912 -0.03499081  0.00905719]]
99.74122619628906


In [9]:
class ScaleRender(gym.Wrapper):
    """Nearest-neighbour upscale of the rendered frame. Does not touch observations."""

    def __init__(self, env, scale=4):
        super().__init__(env)
        self.scale = scale

    def render(self):
        frame = self.env.render()
        if frame is None:
            return frame
        return np.repeat(np.repeat(frame, self.scale, axis=0), self.scale, axis=1)

## Copying parameters to the target network

The TF version built one `assign` op per variable and matched them up by sorted name. In
PyTorch a state dict copy does the whole job, so `ModelParametersCopier` exists here only to
keep the call sites looking the same — `copy_model_parameters(q, target)` is equally fine.

In [10]:
class ModelParametersCopier:
    """
    Copy model parameters of one estimator to another.
    """

    def __init__(self, estimator1, estimator2):
        """
        Args:
          estimator1: Estimator to copy the parameters from
          estimator2: Estimator to copy the parameters to
        """
        self.estimator1 = estimator1
        self.estimator2 = estimator2

    def make(self):
        """Makes copy."""
        self.estimator2.model.load_state_dict(self.estimator1.model.state_dict())

In [11]:
def make_epsilon_greedy_policy(estimator, nA):
    """
    Creates an epsilon-greedy policy based on a given Q-function approximator and epsilon.

    Args:
        estimator: An estimator that returns q values for a given state
        nA: Number of actions in the environment.

    Returns:
        A function that takes (observation, epsilon) as arguments and returns
        the probabilities for each action in the form of a numpy array of length nA.
    """

    def policy_fn(observation, epsilon):
        A = np.ones(nA, dtype=float) * epsilon / nA
        q_values = estimator.predict(np.expand_dims(observation, 0))[0]
        best_action = np.argmax(q_values)
        A[best_action] += (1.0 - epsilon)
        return A

    return policy_fn

## The training loop

Structurally identical to the original. Three small changes beyond dropping `sess`:

- The replay memory is a `deque(maxlen=...)`, so evicting the oldest transition is O(1)
  instead of the O(n) `list.pop(0)` the original did on every step past 500k.
- Frames stack along axis 0 (channels first) rather than axis 2.
- `total_t` increments on every step, including the terminal one. The original `break`'d
  before the increment, so terminal steps were never counted.

In [12]:
def deep_q_learning(env,
                    q_estimator,
                    target_estimator,
                    state_processor,
                    num_episodes,
                    experiment_dir,
                    replay_memory_size=500000,
                    replay_memory_init_size=50000,
                    update_target_estimator_every=10000,
                    discount_factor=0.99,
                    epsilon_start=1.0,
                    epsilon_end=0.1,
                    epsilon_decay_steps=500000,
                    batch_size=32,
                    record_video_every=50):
    """
    Q-Learning algorithm for off-policy TD control using Function Approximation.
    Finds the optimal greedy policy while following an epsilon-greedy policy.

    Args:
        env: OpenAI environment
        q_estimator: Estimator object used for the q values
        target_estimator: Estimator object used for the targets
        state_processor: A StateProcessor object
        num_episodes: Number of episodes to run for
        experiment_dir: Directory to save checkpoints and summaries in
        replay_memory_size: Size of the replay memory
        replay_memory_init_size: Number of random experiences to sample when initializing
          the replay memory.
        update_target_estimator_every: Copy parameters from the Q estimator to the
          target estimator every N steps
        discount_factor: Gamma discount factor
        epsilon_start: Chance to sample a random action when taking an action.
          Epsilon is decayed over time and this is the start value
        epsilon_end: The final minimum value of epsilon after decaying is done
        epsilon_decay_steps: Number of steps to decay epsilon over
        batch_size: Size of batches to sample from the replay memory
        record_video_every: Record a video every N episodes

    Returns:
        An EpisodeStats object with two numpy arrays for episode_lengths and episode_rewards.
    """

    Transition = namedtuple("Transition", ["state", "action", "reward", "next_state", "done"])

    # The replay memory. maxlen gives us O(1) eviction of the oldest transition.
    replay_memory = deque(maxlen=replay_memory_size)

    # Make model copier object
    estimator_copy = ModelParametersCopier(q_estimator, target_estimator)

    # Keeps track of useful statistics
    stats = EpisodeStats(
        episode_lengths=np.zeros(num_episodes),
        episode_rewards=np.zeros(num_episodes))

    # For 'system/' summaries, useful to check if current process looks healthy
    current_process = psutil.Process()

    # Create directories for checkpoints and summaries
    checkpoint_dir = os.path.join(experiment_dir, "checkpoints")
    checkpoint_path = os.path.join(checkpoint_dir, "model.pt")
    monitor_path = os.path.join(experiment_dir, "monitor")

    os.makedirs(checkpoint_dir, exist_ok=True)
    os.makedirs(monitor_path, exist_ok=True)

    # Load a previous checkpoint if we find one
    total_t = 0
    if os.path.exists(checkpoint_path):
        print("Loading model checkpoint {}...\n".format(checkpoint_path))
        ckpt = torch.load(checkpoint_path, map_location=q_estimator.device)
        q_estimator.model.load_state_dict(ckpt["q_estimator"])
        target_estimator.model.load_state_dict(ckpt["target_estimator"])
        q_estimator.optimizer.load_state_dict(ckpt["optimizer"])
        q_estimator.global_step = ckpt["global_step"]
        total_t = ckpt["total_t"]

    # The epsilon decay schedule
    epsilons = np.linspace(epsilon_start, epsilon_end, epsilon_decay_steps)

    # The policy we're following
    policy = make_epsilon_greedy_policy(
        q_estimator,
        len(VALID_ACTIONS))

    # Populate the replay memory with initial experience
    print("Populating replay memory...")
    state = env_reset(env)
    state = state_processor.process(state)
    state = np.stack([state] * 4, axis=0)
    for i in range(replay_memory_init_size):
        action_probs = policy(state, epsilons[min(total_t, epsilon_decay_steps - 1)])
        action = np.random.choice(np.arange(len(action_probs)), p=action_probs)
        next_state, reward, done, _ = env_step(env, VALID_ACTIONS[action])
        next_state = state_processor.process(next_state)
        next_state = np.concatenate([state[1:], next_state[np.newaxis]], axis=0)
        replay_memory.append(Transition(state, action, reward, next_state, done))
        if done:
            state = env_reset(env)
            state = state_processor.process(state)
            state = np.stack([state] * 4, axis=0)
        else:
            state = next_state

    # Record videos. Recording is a nice-to-have, so any failure here degrades to
    # "no videos" rather than throwing away the replay memory we just spent 50k steps
    # filling. Needs the env built with render_mode="rgb_array".
    try:
        try:
            from gymnasium.wrappers import RecordVideo
            env = ScaleRender(env, scale=4)
            env = RecordVideo(env, video_folder=monitor_path,
                              episode_trigger=lambda count: count % record_video_every == 0)
        except ImportError:
            from gym.wrappers import Monitor
            env = Monitor(env, directory=monitor_path,
                          video_callable=lambda count: count % record_video_every == 0,
                          resume=True)
    except (ImportError, ValueError) as err:
        print("\nVideo recording disabled ({}); continuing without it.".format(err))

    for i_episode in range(num_episodes):

        # Save the current checkpoint
        torch.save({
            "q_estimator": q_estimator.model.state_dict(),
            "target_estimator": target_estimator.model.state_dict(),
            "optimizer": q_estimator.optimizer.state_dict(),
            "global_step": q_estimator.global_step,
            "total_t": total_t,
        }, checkpoint_path)

        # Reset the environment
        state = env_reset(env)
        state = state_processor.process(state)
        state = np.stack([state] * 4, axis=0)
        loss = None

        # One step in the environment
        for t in itertools.count():

            # Epsilon for this time step
            epsilon = epsilons[min(total_t, epsilon_decay_steps - 1)]

            # Maybe update the target estimator
            if total_t % update_target_estimator_every == 0:
                estimator_copy.make()
                print("\nCopied model parameters to target network.")

            # Print out which step we're on, useful for debugging.
            if (i_episode+1) % 50 == 0:
                print("\rStep {} ({}) @ Episode {}/{}, loss: {}".format(
                    t, total_t, i_episode + 1, num_episodes, loss), end="")
                sys.stdout.flush()

            # Take a step
            action_probs = policy(state, epsilon)
            action = np.random.choice(np.arange(len(action_probs)), p=action_probs)
            next_state, reward, done, _ = env_step(env, VALID_ACTIONS[action])
            next_state = state_processor.process(next_state)
            next_state = np.concatenate([state[1:], next_state[np.newaxis]], axis=0)

            # Save transition to replay memory (the deque evicts the oldest for us)
            replay_memory.append(Transition(state, action, reward, next_state, done))

            # Update statistics
            stats.episode_rewards[i_episode] += reward
            stats.episode_lengths[i_episode] = t

            # Sample a minibatch from the replay memory
            samples = random.sample(replay_memory, batch_size)
            states_batch, action_batch, reward_batch, next_states_batch, done_batch = map(
                np.array, zip(*samples))

            # Calculate q values and targets
            q_values_next = target_estimator.predict(next_states_batch)
            targets_batch = reward_batch + np.invert(done_batch).astype(np.float32) * \
                discount_factor * np.amax(q_values_next, axis=1)

            # Perform gradient descent update
            loss = q_estimator.update(states_batch, action_batch, targets_batch)

            total_t += 1

            if done:
                break

            state = next_state

        # Add summaries to tensorboard
        writer = q_estimator.summary_writer
        if writer:
            writer.add_scalar("episode/epsilon", epsilon, i_episode)
            writer.add_scalar("episode/reward", stats.episode_rewards[i_episode], i_episode)
            writer.add_scalar("episode/length", stats.episode_lengths[i_episode], i_episode)
            writer.add_scalar("system/cpu_usage_percent",
                              current_process.cpu_percent(), i_episode)
            writer.add_scalar("system/v_memory_usage_percent",
                              current_process.memory_percent(memtype="vms"), i_episode)
            writer.flush()

        yield total_t, EpisodeStats(
            episode_lengths=stats.episode_lengths[:i_episode + 1],
            episode_rewards=stats.episode_rewards[:i_episode + 1])

    return stats

In [13]:
# Where we save our checkpoints and graphs
experiment_dir = os.path.abspath("./experiments/{}".format(env.spec.id.replace("/", "_")))

# Create estimators
q_estimator = Estimator(scope="q_estimator", summaries_dir=experiment_dir)
target_estimator = Estimator(scope="target_q")

# State processor
state_processor = StateProcessor()

# Run it!
for i, (t, stats) in enumerate(deep_q_learning(env,
                                q_estimator=q_estimator,
                                target_estimator=target_estimator,
                                state_processor=state_processor,
                                experiment_dir=experiment_dir,
                                num_episodes=10000,
                                replay_memory_size=100000,
                                replay_memory_init_size=50000,
                                update_target_estimator_every=10000,
                                epsilon_start=1.0,
                                epsilon_end=0.1,
                                epsilon_decay_steps=500000,
                                discount_factor=0.99,
                                batch_size=32)):
    if (i+1) % 50 == 0:
        print("\nEpisode {} Reward: {}".format(i + 1, stats.episode_rewards[-1]))

Populating replay memory...


KeyboardInterrupt: 